In [ ]:
from pathlib import Path

AUDIT_VERSION = "legalir-kaggle-package-audit-v1"
RESULT_PATH = Path("/kaggle/working/kaggle_package_environment_audit.json")

PACKAGE_SPECS = {
    "torch": {"module": "torch", "symbols": []},
    "transformers": {"module": "transformers", "symbols": ["AutoModel", "AutoModelForSequenceClassification", "AutoTokenizer", "GemmaForCausalLM"]},
    "tokenizers": {"module": "tokenizers", "symbols": []},
    "FlagEmbedding": {"module": "FlagEmbedding", "symbols": ["BGEM3FlagModel", "FlagLLMReranker"]},
    "accelerate": {"module": "accelerate", "symbols": []},
    "peft": {"module": "peft", "symbols": []},
    "sentencepiece": {"module": "sentencepiece", "symbols": []},
    "protobuf": {"module": "google.protobuf", "symbols": []},
    "psutil": {"module": "psutil", "symbols": []},
    "bm25s": {"module": "bm25s", "symbols": ["BM25"]},
    "sentence-transformers": {"module": "sentence_transformers", "symbols": ["MultiVectorEncoder", "CrossEncoder"]},
    "safetensors": {"module": "safetensors", "symbols": []},
    "huggingface-hub": {"module": "huggingface_hub", "symbols": []},
    "numpy": {"module": "numpy", "symbols": []},
    "scipy": {"module": "scipy", "symbols": []},
    "scikit-learn": {"module": "sklearn", "symbols": []},
    "datasets": {"module": "datasets", "symbols": []},
    "packaging": {"module": "packaging", "symbols": []},
    "einops": {"module": "einops", "symbols": []},
    "faiss-gpu": {"module": "faiss", "symbols": ["StandardGpuResources", "GpuIndexFlatConfig", "GpuIndexFlatIP"]},
}

PROFILES = {
    "bge_m3_sparse_and_dense_sparse_hybrid": {
        "cuda_required": True,
        "requirements": {
            "torch": {"specifier": None, "symbols": []},
            "transformers": {"specifier": ">=4.44.2,<6", "symbols": ["AutoModel", "AutoTokenizer"]},
            "tokenizers": {"specifier": None, "symbols": []},
            "FlagEmbedding": {"specifier": ">=1.4.2,<2", "symbols": ["BGEM3FlagModel"]},
            "accelerate": {"specifier": ">=0.20.1", "symbols": []},
            "peft": {"specifier": None, "symbols": []},
            "sentencepiece": {"specifier": None, "symbols": []},
            "protobuf": {"specifier": None, "symbols": []},
            "psutil": {"specifier": None, "symbols": []},
            "bm25s": {"specifier": ">=0.3.11,<0.4", "symbols": ["BM25"]},
            "numpy": {"specifier": None, "symbols": []},
            "packaging": {"specifier": None, "symbols": []},
        },
    },
    "bge_reranker_v2_gemma": {
        "cuda_required": True,
        "requirements": {
            "torch": {"specifier": None, "symbols": []},
            "transformers": {"specifier": ">=4.44.2,<6", "symbols": ["AutoTokenizer", "GemmaForCausalLM"]},
            "tokenizers": {"specifier": None, "symbols": []},
            "FlagEmbedding": {"specifier": ">=1.4.2,<2", "symbols": ["FlagLLMReranker"]},
            "accelerate": {"specifier": ">=0.20.1", "symbols": []},
            "peft": {"specifier": None, "symbols": []},
            "sentencepiece": {"specifier": None, "symbols": []},
            "protobuf": {"specifier": None, "symbols": []},
            "psutil": {"specifier": None, "symbols": []},
        },
    },
    "gte_reranker_correctness_audit": {
        "cuda_required": True,
        "requirements": {
            "torch": {"specifier": None, "symbols": []},
            "transformers": {"specifier": ">=4.36,<5", "symbols": ["AutoModelForSequenceClassification", "AutoTokenizer"]},
            "tokenizers": {"specifier": None, "symbols": []},
            "numpy": {"specifier": None, "symbols": []},
            "packaging": {"specifier": None, "symbols": []},
        },
        "optional": {"sentence-transformers": {"specifier": None, "symbols": ["CrossEncoder"]}},
    },
    "jina_colbert_v2_64": {
        "cuda_required": True,
        "requirements": {
            "torch": {"specifier": None, "symbols": []},
            "transformers": {"specifier": ">=4.44.2,<6", "symbols": ["AutoModel", "AutoTokenizer"]},
            "sentence-transformers": {"specifier": ">=6,<7", "symbols": ["MultiVectorEncoder"]},
            "einops": {"specifier": ">=0.8.2", "symbols": []},
            "numpy": {"specifier": None, "symbols": []},
            "scipy": {"specifier": None, "symbols": []},
            "scikit-learn": {"specifier": None, "symbols": []},
            "faiss-gpu": {"specifier": ">=1.15,<1.16", "symbols": ["StandardGpuResources", "GpuIndexFlatConfig", "GpuIndexFlatIP"]},
        },
    },
}


In [ ]:
import importlib
import importlib.metadata
import importlib.util
import json
import os
import platform
import sys
from collections import defaultdict

os.environ.update({
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})


def distribution_version(name: str):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def audit_package(distribution: str, spec: dict) -> dict:
    module_name = spec["module"]
    try:
        module_spec_found = importlib.util.find_spec(module_name) is not None
        spec_error = None
    except Exception as exc:
        module_spec_found = False
        spec_error = f"{type(exc).__name__}: {exc}"
    imported = False
    import_error = spec_error
    symbols = {name: False for name in spec["symbols"]}
    module_version = None
    if module_spec_found:
        try:
            module = importlib.import_module(module_name)
            imported = True
            import_error = None
            module_version = getattr(module, "__version__", None)
            symbols = {name: hasattr(module, name) for name in spec["symbols"]}
        except Exception as exc:
            import_error = f"{type(exc).__name__}: {exc}"
    return {
        "distribution": distribution,
        "module": module_name,
        "installed_version": distribution_version(distribution),
        "module_version": str(module_version) if module_version is not None else None,
        "module_spec_found": module_spec_found,
        "import_ok": imported,
        "import_error": import_error,
        "symbols": symbols,
    }


def version_satisfies(version: str | None, specifier: str | None):
    if version is None:
        return False
    if specifier is None:
        return True
    try:
        from packaging.specifiers import SpecifierSet
        from packaging.version import Version
        return Version(version) in SpecifierSet(specifier)
    except Exception:
        return None


def audit_requirement(name: str, requirement: dict, packages: dict) -> dict:
    package = packages[name]
    version_ok = version_satisfies(package["installed_version"], requirement["specifier"])
    required_symbols = requirement.get("symbols", [])
    missing_symbols = [symbol for symbol in required_symbols if not package["symbols"].get(symbol, False)]
    return {
        "specifier": requirement["specifier"],
        "installed_version": package["installed_version"],
        "installed": package["installed_version"] is not None,
        "version_ok": version_ok,
        "import_ok": package["import_ok"],
        "import_error": package["import_error"],
        "missing_symbols": missing_symbols,
        "passed": bool(version_ok is True and package["import_ok"] and not missing_symbols),
    }


def audit_profile(name: str, profile: dict, packages: dict, cuda_available: bool) -> dict:
    requirements = {
        package: audit_requirement(package, requirement, packages)
        for package, requirement in profile["requirements"].items()
    }
    optional = {
        package: audit_requirement(package, requirement, packages)
        for package, requirement in profile.get("optional", {}).items()
    }
    missing = [package for package, item in requirements.items() if not item["installed"]]
    incompatible = [
        package for package, item in requirements.items()
        if item["installed"] and item["version_ok"] is not True
    ]
    import_failures = [package for package, item in requirements.items() if item["installed"] and not item["import_ok"]]
    api_failures = [package for package, item in requirements.items() if item["missing_symbols"]]
    package_ready = all(item["passed"] for item in requirements.values())
    cuda_ready = cuda_available or not profile["cuda_required"]
    return {
        "profile": name,
        "requirements": requirements,
        "optional": optional,
        "missing_packages": missing,
        "incompatible_packages": incompatible,
        "import_failures": import_failures,
        "api_failures": api_failures,
        "packages_ready_without_install": package_ready,
        "cuda_required": profile["cuda_required"],
        "cuda_ready": cuda_ready,
        "ready_to_run": package_ready and cuda_ready,
        "recommended_downloads": sorted(set(missing + incompatible + import_failures + api_failures)),
    }


def installed_distribution_inventory() -> dict:
    inventory = defaultdict(set)
    for distribution in importlib.metadata.distributions():
        name = distribution.metadata.get("Name")
        if name:
            inventory[name].add(distribution.version)
    return {name: sorted(versions) for name, versions in sorted(inventory.items(), key=lambda item: item[0].lower())}


In [ ]:
packages = {name: audit_package(name, spec) for name, spec in PACKAGE_SPECS.items()}
torch_report = packages["torch"]
cuda = {"available": False, "device_count": 0, "devices": [], "torch_cuda_version": None}
if torch_report["import_ok"]:
    import torch
    cuda["available"] = bool(torch.cuda.is_available())
    cuda["device_count"] = int(torch.cuda.device_count())
    cuda["devices"] = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
    cuda["torch_cuda_version"] = torch.version.cuda
profiles = {
    name: audit_profile(name, profile, packages, cuda["available"])
    for name, profile in PROFILES.items()
}
result = {
    "audit_version": AUDIT_VERSION,
    "scope": "Kaggle base-environment package availability only; no installs, downloads, model loads, or dataset reads",
    "environment": {
        "python": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "machine": platform.machine(),
        "cuda": cuda,
        "offline_flags": {name: os.environ[name] for name in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE")},
    },
    "relevant_packages": packages,
    "profiles": profiles,
    "recommended_downloads_union": sorted({
        package
        for profile in profiles.values()
        for package in profile["recommended_downloads"]
    }),
    "installed_distribution_inventory": installed_distribution_inventory(),
}
RESULT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("Relevant package audit:")
for name, package in packages.items():
    state = "OK" if package["import_ok"] else "MISSING" if package["installed_version"] is None else "IMPORT_FAIL"
    print(f"{name:24} {str(package['installed_version']):14} {state}")
print("\nProfile readiness:")
for name, profile in profiles.items():
    print(f"{name:42} packages_ready={profile['packages_ready_without_install']} cuda_ready={profile['cuda_ready']} downloads={profile['recommended_downloads']}")
print("\nRecommended downloads union:", result["recommended_downloads_union"])
print("Saved:", RESULT_PATH)
